# VBC Incentive Alignment - interactive explorer

Drag sliders, click **Run simulation**, and see the per-contract summary plus the plots inline. No install: this runs in Google Colab.

**How to use**
1. Run the **Setup** cell once (clones the repo + installs deps).
2. Run the **Explorer** cell to get the sliders.
3. Adjust parameters, click *Run simulation*. Tick the GIF box for the animated replay.

The repo stays the source of truth; this notebook is just a friendly front door. Parameter meanings and rationale are in `MODEL.md`.


In [ ]:
# Setup: clone the repo + install deps (run once per session).
import os

REPO   = "brendancruz/hc-system-of-record"
BRANCH = "main"   # the branch this notebook clones from

# If the repo is PRIVATE: set a GitHub token ONCE via Colab Secrets (key icon, left
# sidebar). Name it GH_TOKEN, value = a personal access token with read access to
# the repo. Public repo? You can ignore this - it clones without a token.
token = ""
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN") or ""
except Exception:
    pass

auth = f"{token}@" if token else ""
url  = f"https://{auth}github.com/{REPO}.git"

if not os.path.isdir("/content/repo"):
    rc = os.system(f"git clone --depth 1 --branch {BRANCH} {url} /content/repo")
    if rc != 0:
        raise RuntimeError(
            "Clone failed. If the repo is private, add a GH_TOKEN Colab secret "
            "(key icon in the left sidebar) and re-run this cell."
        )

%cd /content/repo/vbc-incentive-sim
!pip -q install -r requirements.txt
print("\nSetup done. Run the next cell to launch the explorer.")

In [ ]:
# VBC simulation explorer - drag sliders, click Run.
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/content/repo/vbc-incentive-sim")

import ipywidgets as W
from IPython.display import display, Image, clear_output

from vbc_sim.config import SimConfig
from vbc_sim.simulation import run_all
from vbc_sim.plotting import make_all_plots
from vbc_sim.animation import animate_simulation

defaults = SimConfig()
STYLE = {"description_width": "160px"}
LAYOUT = W.Layout(width="440px")

def fs(desc, val, lo, hi, step):
    return W.FloatSlider(description=desc, value=val, min=lo, max=hi, step=step,
                         continuous_update=False, readout_format=".2f",
                         style=STYLE, layout=LAYOUT)

w = {
    "alpha":       fs("alpha (health slope)",      defaults.alpha, 0.2, 2.0, 0.05),
    "beta":        fs("beta (over-tx harm)",       defaults.beta, 0.1, 1.5, 0.05),
    "c_eff":       fs("c_eff (cost of effort)",    defaults.c_eff, 0.0, 0.95, 0.05),
    "gamma":       fs("gamma (effort cost)",       defaults.gamma, 0.05, 1.0, 0.05),
    "mu":          fs("mu (intrinsic motivation)", defaults.mu, 0.0, 1.5, 0.05),
    "rho":         fs("rho (risk aversion)",       defaults.rho, 0.0, 5.0, 0.25),
    "phi1":        fs("phi1 (FFS fee)",            defaults.phi1, 0.0, 1.5, 0.05),
    "s_share":     fs("s_share (savings rate)",    defaults.s_share, 0.0, 1.0, 0.05),
    "b_err":       fs("b_err (benchmark level)",   defaults.b_err, -0.5, 0.5, 0.05),
    "b_err_slope": fs("b_err_slope (sick adj.)",   defaults.b_err_slope, -1.0, 1.0, 0.05),
    "sigma_c":     fs("sigma_c (cost noise)",      defaults.sigma_c, 0.0, 1.0, 0.05),
}
w_n = W.Dropdown(description="patients", options=[5000, 20000, 50000, 100000],
                 value=20000, style=STYLE, layout=LAYOUT)
w_contracts = W.SelectMultiple(description="contracts",
                 options=["ffs", "capitation", "shared_savings", "two_sided"],
                 value=("ffs", "capitation", "shared_savings"),
                 style=STYLE, layout=W.Layout(width="440px", height="100px"))
w_gif = W.Checkbox(description="also render animation GIF (slower, ~30s)", value=False)
run_btn = W.Button(description="Run simulation", button_style="success", icon="play")
reset_btn = W.Button(description="Reset to defaults", icon="refresh")
out = W.Output()

COLS = ["accept_rate", "efficiency_ratio", "mean_surplus", "welfare_gap",
        "mean_income", "sd_income", "over_treat_rate", "under_treat_rate",
        "mean_effort_gap", "cream_skim_index"]

def on_run(_):
    with out:
        clear_output(wait=True)
        contracts = list(w_contracts.value)
        if not contracts:
            print("Pick at least one contract (ctrl/cmd-click to multi-select).")
            return
        cfg = defaults.replace(
            **{k: v.value for k, v in w.items()},
            n_patients=int(w_n.value), contracts=contracts,
        )
        print(f"Running {cfg.n_patients:,} patients x {len(contracts)} contracts ...")
        summary, details = run_all(cfg)
        clear_output(wait=True)
        print("e*(s) = %.3f * s   (first-best intensity)" % ((cfg.alpha - cfg.c_eff) / (2 * cfg.beta)))
        display(summary[COLS].round(3))
        os.makedirs("outputs", exist_ok=True)
        for p in make_all_plots(summary, details, cfg, "outputs"):
            display(Image(p))
        if w_gif.value:
            print("Rendering animation ...")
            display(Image(animate_simulation(details, cfg, "outputs/simulation.gif")))

def on_reset(_):
    for k, v in w.items():
        v.value = getattr(defaults, k)
    w_n.value = 20000
    w_contracts.value = ("ffs", "capitation", "shared_savings")
    w_gif.value = False

run_btn.on_click(on_run)
reset_btn.on_click(on_reset)

controls = W.VBox(list(w.values()) + [w_n, w_contracts, w_gif,
                  W.HBox([run_btn, reset_btn])])
display(W.VBox([controls, out]))
print("Drag the sliders, then click Run simulation. (See MODEL.md for what each parameter means.)")